# Prevención de Riesgos de Salud · Región Metropolitana de Guatemala
## Procesamiento Satelital Copernicus Sentinel-2 L2A con Google Earth Engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/colon2025mun-commits/urbano-gt-salud-metropolitana-1/blob/mejora/actualizacion-metodologia/notebooks/analisis_sentinel2_salud_metropolitana.ipynb)

**Creador del proyecto:** Andres Barrionuevo  
**Repositorio:** [copernicuscentroamerica/urbano-gt-salud-metropolitana](https://github.com/copernicuscentroamerica/urbano-gt-salud-metropolitana)  

### Parámetros de corrida verificados:
- **Colección:** Sentinel-2 L2A Armonizada (`COPERNICUS/S2_SR_HARMONIZED`)
- **Umbral máximo de nubosidad:** 20 %
- **Enmascaramiento SCL:** Clases 4 (Vegetación), 5 (Suelo desnudo/no vegetado) y 6 (Agua)
- **Ámbito territorial:** Región metropolitana de Guatemala (Superficie ~2,253 km²; Centroide: 14°37'16.48"N 90°20'51.03"W)

In [ ]:
# Paso 1: Instalación de dependencias en Google Colab
!pip install -q earthengine-api geemap folium

In [ ]:
# Paso 2: Autenticación e Inicialización de Google Earth Engine
import ee
import geemap

try:
    ee.Initialize()
    print("✓ Sesión activa de Earth Engine detectada.")
except Exception:
    print("Iniciando autenticación interactiva con tu cuenta de Google...")
    ee.Authenticate()
    ee.Initialize()
    print("✓ Conexión establecida con Google Earth Engine.")

In [ ]:
# Paso 3: Definición del Área de Estudio y Enmascaramiento SCL
# Coordenadas del centroide declarado: 14°37'16.48"N, 90°20'51.03"W
centroide_lon = -90.347508
centroide_lat = 14.621244
centroide = ee.Geometry.Point([centroide_lon, centroide_lat])

# Buffer de 26.8 km para cubrir los ~2,253 km2 del área metropolitana
roi = centroide.buffer(26800).bounds()

def mask_scl_sentinel2(image):
    """Filtra nubes y sombras conservando clases SCL 4, 5 y 6."""
    scl = image.select('SCL')
    # 4: Vegetación, 5: Suelo no vegetado, 6: Cuerpos de agua
    mask = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6))
    return image.updateMask(mask).divide(10000).copyProperties(image, ['system:time_start'])

# Colección Sentinel-2 con umbral de 20% de nubes
s2_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate('2026-01-01', '2026-11-10')
    .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mask_scl_sentinel2)
)

# Mosaico libre de nubes (mediana temporal)
mosaico = s2_collection.median().clip(roi)
print(f"Imágenes Sentinel-2 procesadas: {s2_collection.size().getInfo()}")

In [ ]:
# Paso 4: Cálculo de Índices Multiespectrales para Salud y Territorio
# NDVI: Exposición a vegetación y áreas verdes
ndvi = mosaico.normalizedDifference(['B8', 'B4']).rename('NDVI')

# NDBI: Densidad edificada / Expansión urbana (indicador de calor urbano)
ndbi = mosaico.normalizedDifference(['B11', 'B8']).rename('NDBI')

# MNDWI: Cuerpos de agua y susceptibilidad a inundación
mndwi = mosaico.normalizedDifference(['B3', 'B11']).rename('MNDWI')

In [ ]:
# Paso 5: Visualización Interactiva en Mapa (Geemap / MapLibre)
Map = geemap.Map(center=[centroide_lat, centroide_lon], zoom=11)

# Parámetros de visualización
vis_rgb = {'bands': ['B4', 'B3', 'B2'], 'min': 0.0, 'max': 0.3}
vis_ndvi = {'min': -0.1, 'max': 0.8, 'palette': ['#d73027', '#f46d43', '#fdae61', '#fee08b', '#a6d96a', '#1a9850']}
vis_ndbi = {'min': -0.3, 'max': 0.3, 'palette': ['#313695', '#74add1', '#e0f3f8', '#fee090', '#f46d43', '#a50026']}

Map.addLayer(mosaico, vis_rgb, 'Sentinel-2 RGB (Color Natural)')
Map.addLayer(ndvi, vis_ndvi, 'NDVI (Vegetación y Salud)')
Map.addLayer(ndbi, vis_ndbi, 'NDBI (Expansión Urbana / Calor)')
Map.add_layer_control()
Map

In [ ]:
# Paso 6: Exportación de Capa para el Geovisor Local (GeoTIFF / Google Drive)
task = ee.batch.Export.image.toDrive(
    image=ndvi.visualize(**vis_ndvi),
    description='gt_salud_metropolitana_ndvi_2026',
    folder='Copernicus_Urbano_GT',
    fileNamePrefix='ndvi_metropolitano_gt_2026',
    region=roi,
    scale=20,
    crs='EPSG:4326',
    maxPixels=1e9
)
# Descomentar para iniciar exportación a Google Drive:
# task.start()
# print("Tarea de exportación iniciada en Google Drive.")